## Changelog
- parent: 20260508_190144_fe3cc645
- change: add a fixed 20% holdout split (random_state=0) carved out before any CV. Pipelines train on the 80% slice with the same KFold(5, shuffle, rs=42); CV produces OOF predictions over the 80%, the Ridge meta is fit on those, and the final blend is evaluated on the untouched 20% holdout. After the holdout score is reported, the entire stack is refit on 100% of the training data for the test-set submission. The submission predictions are unchanged in expectation; the holdout score is the new oracle.
- hypothesis: five documented CV-LB inversions across 5-7/5-8 (+154, +156, +−138, +−128, +−975) say the in-fold CV is not a reliable filter at this dataset size. A held-out 20% split (~290 rows) gives a single scalar that has not been seen by any model selection step in the session, and so calibrates LB expectation more honestly than CV alone. Cost: every change loses 20% training data for evaluation purposes; benefit: catches the runs where CV says go and LB says no before we burn a daily submission slot.


In [ ]:
import sys
import numpy as np
import pandas as pd

from pathlib import Path

IS_KAGGLE = Path("/kaggle/input").exists()

if IS_KAGGLE:
    data_dir   = Path("/kaggle/input/home-data-for-ml-course")
    output_dir = Path("/kaggle/working")
else:
    def _find_competition_dir(start: Path) -> Path:
        for p in [start, *start.parents]:
            if (p / "config.yaml").exists() and (p / "data").is_dir():
                return p
        raise RuntimeError("competition dir not found (no ancestor has config.yaml + data/)")

    comp_dir   = _find_competition_dir(Path.cwd())
    data_dir   = comp_dir / "data"
    output_dir = Path.cwd()

    if str(comp_dir) not in sys.path:
        sys.path.insert(0, str(comp_dir))

train_data_raw = pd.read_csv(data_dir / "train.csv")
test_data_raw  = pd.read_csv(data_dir / "test.csv")


In [ ]:
# see eda-TotalSF.ipynb — drop the mega-house outliers (TotalSF > 7000)
_outlier_mask = (
    train_data_raw["TotalBsmtSF"]
    + train_data_raw["1stFlrSF"]
    + train_data_raw["2ndFlrSF"]
) > 7000
train_data = train_data_raw.loc[~_outlier_mask].reset_index(drop=True)

DROP = ["Id", "SalePrice"]
X      = train_data.drop(columns=DROP, errors="ignore").copy()
X_test = test_data_raw.drop(columns=DROP, errors="ignore").copy()
y      = np.log1p(train_data["SalePrice"])

# MSSubClass is a nominal int code — cast to string so the encoder treats it
# as a category rather than an ordered number.
X["MSSubClass"]      = X["MSSubClass"].astype(str)
X_test["MSSubClass"] = X_test["MSSubClass"].astype(str)

NUMERIC     = X.select_dtypes(include="number").columns.tolist()
CATEGORICAL = X.select_dtypes(exclude="number").columns.tolist()
print(f"{len(NUMERIC)} numeric + {len(CATEGORICAL)} categorical = {len(X.columns)} total")


In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin


class CategoryCaster(BaseEstimator, TransformerMixin):
    """Pin categorical vocabulary at fit-time, replay at transform."""

    def fit(self, X, y=None):
        self.vocab_ = {}
        cat_like = X.select_dtypes(include=["object", "string", "category"]).columns
        for c in cat_like:
            self.vocab_[c] = sorted(X[c].dropna().astype(str).unique())
        return self

    def transform(self, X):
        out = X.copy()
        for c, cats in self.vocab_.items():
            if c in out.columns:
                out[c] = pd.Categorical(out[c].astype(object), categories=cats)
        return out


class BoxCoxSkewed(BaseEstimator, TransformerMixin):
    """boxcox1p(lam) on numeric columns with |skew| > threshold (per-fold)."""

    def __init__(self, threshold: float = 0.75, lam: float = 0.15):
        self.threshold = threshold
        self.lam = lam

    def fit(self, X, y=None):
        from scipy.stats import skew
        numeric = X.select_dtypes(include="number").columns
        skews = X[numeric].apply(lambda s: skew(s.dropna()))
        self.skewed_cols_ = skews[skews.abs() > self.threshold].index.tolist()
        return self

    def transform(self, X):
        from scipy.special import boxcox1p
        out = X.copy()
        for c in self.skewed_cols_:
            if c in out.columns:
                out[c] = boxcox1p(out[c], self.lam)
        return out


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, RobustScaler
from sklearn.linear_model import Lasso, Ridge
from sklearn.kernel_ridge import KernelRidge
from sklearn.model_selection import KFold

from xgboost import XGBRegressor

from utils.ames_sklearn_pipeline import AmesNAImputer, AmesEncoder
from utils.ames_feature_engineering import (
    add_size_features, add_temporal_features, add_bath_features,
    add_quality_size_features,
    add_porch_features, add_presence_indicators, add_ratio_features,
)

class SeedBag(BaseEstimator, TransformerMixin):
    """Sklearn-compatible meta-estimator that fits one estimator per seed and
    averages predictions. The wrapped estimator must accept a `random_state`
    kw via clone+set_params.

    Implements predict only (no transform) but inherits TransformerMixin to
    keep the BaseEstimator surface consistent with the other classes here.
    """

    def __init__(self, estimator, seeds=(42, 1, 7)):
        self.estimator = estimator
        self.seeds = tuple(seeds)

    def fit(self, X, y=None):
        from sklearn.base import clone as _clone
        self.estimators_ = []
        for s in self.seeds:
            est = _clone(self.estimator).set_params(random_state=s)
            est.fit(X, y)
            self.estimators_.append(est)
        return self

    def predict(self, X):
        return np.mean([e.predict(X) for e in self.estimators_], axis=0)

xgb_pipe = Pipeline([
    ("na",            AmesNAImputer()),
    ("quality_size",  FunctionTransformer(
                          add_quality_size_features,
                          kw_args={"drop_originals": False},
                      )),
    ("bath",          FunctionTransformer(
                          add_bath_features,
                          kw_args={"drop_originals": False},
                      )),
    ("presence",      FunctionTransformer(add_presence_indicators)),
    ("porch",         FunctionTransformer(
                          add_porch_features,
                          kw_args={"drop_originals": False},
                      )),
    ("ratio",         FunctionTransformer(add_ratio_features)),
    ("size",          FunctionTransformer(
                          add_size_features,
                          kw_args={"drop_originals": True},
                      )),
    ("fe",            FunctionTransformer(
                          add_temporal_features,
                          kw_args={"drop_originals": True},
                      )),
    ("cat_cast",      CategoryCaster()),
    ("model",         SeedBag(
                          XGBRegressor(
                              tree_method="hist",
                              enable_categorical=True,
                              learning_rate=0.05,
                              n_estimators=600,
                              max_depth=4,
                              min_child_weight=1,
                              reg_lambda=1,
                              subsample=0.8,
                              colsample_bytree=0.8,
                              n_jobs=1,
                              verbosity=0,
                          ),
                          seeds=(42, 1, 7),
                      )),
])

from sklearn.ensemble import HistGradientBoostingRegressor

hgbr_pipe = Pipeline([
    ("na",            AmesNAImputer()),
    ("quality_size",  FunctionTransformer(
                          add_quality_size_features,
                          kw_args={"drop_originals": False},
                      )),
    ("bath",          FunctionTransformer(
                          add_bath_features,
                          kw_args={"drop_originals": False},
                      )),
    ("presence",      FunctionTransformer(add_presence_indicators)),
    ("porch",         FunctionTransformer(
                          add_porch_features,
                          kw_args={"drop_originals": False},
                      )),
    ("ratio",         FunctionTransformer(add_ratio_features)),
    ("size",          FunctionTransformer(
                          add_size_features,
                          kw_args={"drop_originals": True},
                      )),
    ("fe",            FunctionTransformer(
                          add_temporal_features,
                          kw_args={"drop_originals": True},
                      )),
    ("cat_cast",  CategoryCaster()),
    ("model",     HistGradientBoostingRegressor(
                      max_iter=600,
                      learning_rate=0.05,
                      max_leaf_nodes=31,
                      min_samples_leaf=20,
                      l2_regularization=1.0,
                      categorical_features="from_dtype",
                      random_state=42,
                  )),
])

def make_linear_pipe(model):
    return Pipeline([
        ("na",            AmesNAImputer()),
        ("quality_size",  FunctionTransformer(
                              add_quality_size_features,
                              kw_args={"drop_originals": False},
                          )),
        ("bath",          FunctionTransformer(
                              add_bath_features,
                              kw_args={"drop_originals": False},
                          )),
        ("presence",      FunctionTransformer(add_presence_indicators)),
        ("porch",         FunctionTransformer(
                              add_porch_features,
                              kw_args={"drop_originals": False},
                          )),
        ("ratio",         FunctionTransformer(add_ratio_features)),
        ("size",          FunctionTransformer(
                              add_size_features,
                              kw_args={"drop_originals": True},
                          )),
        ("fe",            FunctionTransformer(
                              add_temporal_features,
                              kw_args={"drop_originals": True},
                          )),
        ("skew",      BoxCoxSkewed(threshold=0.75, lam=0.15)),
        ("encoder",   AmesEncoder()),
        ("scaler",    RobustScaler()),
        ("model",     model),
    ])

lasso_pipe = make_linear_pipe(Lasso(alpha=0.0005, random_state=1, max_iter=10000))
ridge_pipe = make_linear_pipe(Ridge(alpha=10, random_state=2))
krr_pipe   = make_linear_pipe(KernelRidge(alpha=0.6, kernel="polynomial", degree=2, coef0=2.5))

BASE_PIPES = {
    "XGBoost": xgb_pipe,
    "HGBR": hgbr_pipe,
    "Lasso": lasso_pipe,
    "Ridge": ridge_pipe,
    "KRR": krr_pipe,
}


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.base import clone

X_tr, X_ho, y_tr, y_ho = train_test_split(
    X, y, test_size=0.20, random_state=0
)
print(f"train: {len(X_tr)} rows | holdout: {len(X_ho)} rows")

cv = KFold(n_splits=5, shuffle=True, random_state=42)


def oof_predict(pipe, X_, y_, cv):
    oof = np.zeros(len(X_))
    for tr_idx, vl_idx in cv.split(X_):
        p = clone(pipe)
        p.fit(X_.iloc[tr_idx], y_.iloc[tr_idx])
        oof[vl_idx] = p.predict(X_.iloc[vl_idx])
    return oof


print("Generating OOF predictions per base model on the 80% train slice...")
oof = {}
for name, pipe in BASE_PIPES.items():
    oof[name] = oof_predict(pipe, X_tr, y_tr, cv)
    rmse = np.sqrt(np.mean((oof[name] - y_tr.values) ** 2))
    print(f"  {name:8s}: 5-fold OOF RMSE (log) = {rmse:.4f}")


oof_matrix = np.column_stack([oof[n] for n in BASE_PIPES])
eq_blend_oof = oof_matrix.mean(axis=1)
eq_oof_rmse = np.sqrt(np.mean((eq_blend_oof - y_tr.values) ** 2))
print(f"\nEqual-weight OOF RMSE (log, train slice): {eq_oof_rmse:.4f}")


from sklearn.linear_model import Ridge as _MetaRidge

meta = _MetaRidge(alpha=1.0, fit_intercept=True, positive=True)
meta.fit(oof_matrix, y_tr)
print("\nMeta Ridge weights (positive-constrained):")
for n, w in zip(BASE_PIPES, meta.coef_):
    print(f"  {n:8s}: {w:.3f}")
print(f"  intercept = {meta.intercept_:.3f}")

stacked_oof = meta.predict(oof_matrix)
stacked_oof_rmse = np.sqrt(np.mean((stacked_oof - y_tr.values) ** 2))
print(f"Stacked OOF RMSE (log, train slice): {stacked_oof_rmse:.4f}")


# ---- Refit each base on the full 80% slice, then evaluate on the 20% holdout
print("\nRefitting base models on 80% slice and scoring on holdout...")
for pipe in BASE_PIPES.values():
    pipe.fit(X_tr, y_tr)

ho_preds_log = np.column_stack([
    pipe.predict(X_ho) for pipe in BASE_PIPES.values()
])
eq_ho_log = ho_preds_log.mean(axis=1)
eq_ho_rmse = np.sqrt(np.mean((eq_ho_log - y_ho.values) ** 2))

stacked_ho_log = meta.predict(ho_preds_log)
stacked_ho_rmse = np.sqrt(np.mean((stacked_ho_log - y_ho.values) ** 2))

print(f"Holdout RMSE (log) — equal-weight : {eq_ho_rmse:.4f}")
print(f"Holdout RMSE (log) — stacked Ridge: {stacked_ho_rmse:.4f}")

# Approximate dollar-RMSE on the holdout (LB-comparable).
eq_ho_dollar = np.sqrt(np.mean((np.expm1(eq_ho_log) - np.expm1(y_ho.values)) ** 2))
stacked_ho_dollar = np.sqrt(np.mean((np.expm1(stacked_ho_log) - np.expm1(y_ho.values)) ** 2))
print(f"Holdout RMSE ($) — equal-weight : {eq_ho_dollar:,.2f}")
print(f"Holdout RMSE ($) — stacked Ridge: {stacked_ho_dollar:,.2f}")


# ---- Refit on 100% data for the actual submission. The meta-Ridge is kept
# from the train-slice fit (re-fitting on 100% would require re-generating
# OOF on 100%, which is what the CV already gave us).
print("\nRefitting base models on full 100% data for submission...")
for pipe in BASE_PIPES.values():
    pipe.fit(X, y)


In [ ]:
test_logs = np.column_stack([
    pipe.predict(X_test) for pipe in BASE_PIPES.values()
])
test_log = meta.predict(test_logs)
test_pred = np.expm1(test_log)

sample = pd.read_csv(data_dir / "sample_submission.csv")
submission = sample.copy()
submission["SalePrice"] = test_pred
submission.to_csv(output_dir / "submission.csv", index=False)
print("submission.csv written.")
